In [1]:
#Installs on colab
def running_on_colab():
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False

if running_on_colab():
    print("Running on Colab. Installing dependencies...")
    !pip install -q atomai neuraloperator ipympl ipywidgets hoffmanstmpy

    from google.colab import output
    output.enable_custom_widget_manager()

    !git clone -b ayush_dev --single-branch https://github.com/gnganesh99/NN_Error.git
    %cd NN_Error

    print("Colab setup complete.")

Running on Colab. Installing dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 kB 19.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.0/179.0 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.6/248.6 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.0/519.0 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.3/145.3 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.2/291.2 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 120.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 140.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.8/174.8 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

In [2]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset
import os
from scipy.ndimage import zoom
import pandas as pd 


try:    
    get_ipython().run_line_magic("matplotlib", "widget")
except (ValueError, RuntimeError):
    get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt


# Make package and notebook-local helper modules importable whether this starts in repo root or notebooks/
repo_root = Path.cwd() if (Path.cwd() / "src" / "nnerror").exists() else Path.cwd().parent.parent
src_root = str((repo_root / "src").resolve())
nb_root = str((repo_root / "notebooks"/"STM").resolve())
for import_root in (src_root, nb_root):
    if import_root not in sys.path:
        sys.path.insert(0, import_root)
        

from nnerror.networks import FNO_im2spec, CustomDecoder, im2spec, im2spec_attn, FNO_im2spec_attn
from nnerror.networks.custom_nn import CustomCNN
from nnerror.training_functions import err_estimation, train_model, predict_spectra, norm_0to1, predict_posterior, append_training_set
from nnerror.plot_functions import plot_only_training_loss, plot_training_loss, \
    plot_error_prediction, plot_spectra, plot_error_prediction_3d, plot_scale_slider, plot_latent_space, cluster_latent_space, plot_latent_distribution, plot_scale_specific_error_and_acquisition_maps
from nnerror.utils import append_multiscale_data, edges_zeroed_image, interpolated_center_crop, paired_images_spectra_1
from mpl_toolkits.axes_grid1 import make_axes_locatable

In [ ]:
import os, h5py, numpy as np, matplotlib.pyplot as plt
from stm_utils import Sxm_Image
from nnerror.training_functions import norm_0to1

# Path to the saved results inside notebooks/STM
results_folder = nb_root/"data"/"active_learning_results_ms1"

# List available files to confirm
print("Available files:", os.listdir(results_folder))

# Pick the correct file (adjust if needed)
h5_file = os.path.join(
    results_folder,
    "al_im2spec_L1_WS32_latent5_appendpad_20iters_5pts_distanceacq.h5"
)

# Load ragged HDF5 results
def load_h5_ragged(filename):
    data_dict = {}
    with h5py.File(filename, "r") as f:
        for key in f.keys():
            group = f[key]
            values = []
            for i in range(len(group)):
                values.append(group[str(i)][()])
            data_dict[key] = values
    return data_dict

output_dict = load_h5_ragged(h5_file)

# Extract acquired points
initial_train_indices = np.array(output_dict["train_indices"][0])
final_train_indices = np.array(output_dict["train_indices"][-1])
acquired_indices = np.setdiff1d(final_train_indices, initial_train_indices)

reduced_coords = np.array(output_dict["reduced_coords"][-1])
acquired_coords = reduced_coords[acquired_indices]

# Load STM morphology image
img_sxm = "/content/NN_Error/notebooks/STM/data/large_area/EnZn2As2017.sxm"
full_image = Sxm_Image(img_sxm).image()
full_image = norm_0to1(full_image)

# Plot acquisitions
plt.figure(figsize=(7, 7))
plt.imshow(full_image, cmap="gray", origin="lower")
plt.scatter(
    acquired_coords[:, 0],
    acquired_coords[:, 1],
    c="red",
    s=40,
    edgecolors="white",
    linewidths=0.8,
    label="Acquired points"
)
plt.legend()
plt.title("Active Learning Acquired Points (STM Data)")
plt.axis("off")
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: '/content/NN_Error/notebooks/STM/data/active_learning_results_ms1'